In [ ]:
import math
import argparse
from tqdm import tqdm

import torch
import torchvision
import matplotlib.pyplot as plt
from torchvision.transforms.v2 import ToTensor
from sklearn.model_selection import train_test_split

from MAE.model import *
from src import dataset
from src import configs as cfg
from src import metrics
from MAE.utils import setup_seed


def eval_model(model: torch.nn.Module, valid_loader):
    ''' visualize the first 16 predicted images on val dataset'''
    model.eval()
    with torch.no_grad():
        val_img, val_mask = next(iter(valid_loader))
        # print(len(val_img))
        # val_img = torch.stack([val_dataset[i][0] for i in range(16)])
        val_img = val_img.to(device)
        predicted_val_img, mask = model(val_img)
        predicted_val_img = predicted_val_img * mask + val_img * (1 - mask)
        img = torch.cat([val_img * (1 - mask), predicted_val_img, val_img], dim=0)
        img = rearrange(img, '(v h1 w1) c h w -> c (h1 h) (w1 v w)', w1=2, v=3)
        plt.imshow(img.detach().cpu().numpy().swapaxes(0, 2))
        plt.show()
        # print()
        # loss = torch.mean((predicted_val_img - img) ** 2 * mask) / mask_ratio
        # print("validation loss:", loss.item())


if __name__ == '__main__':
    seed=42
    batch_size=512
    base_learning_rate=2e-4
    weight_decay=0.05
    mask_ratio=0.75
    total_epoch=50
    warmup_epoch=10
    model_path='vit-t-mae.pt'

    setup_seed(seed)
    torch.backends.cuda.matmul.fp32_precision = 'ieee'
    batch_size = load_batch_size = batch_size

    assert batch_size % load_batch_size == 0
    steps_per_update = batch_size // load_batch_size

    # train_dataset = torchvision.datasets.CIFAR10('data', train=True, download=True, transform=ToTensor())
    # val_dataset = torchvision.datasets.CIFAR10('data', train=False, download=True, transform=ToTensor())
    dataset.mk_dataset(verbose=False)
    train_cfg = cfg.TrainingConfig(batch_size=64)
    model_cfg = cfg.ModelConfig()
    dataset_cfg = cfg.DatasetConfig()

    x_train, y_train, x_test = dataset.load_preprocessed_dataset()
    x_train, x_valid, y_train, y_valid = train_test_split(
        x_train,
        y_train,
        test_size=dataset_cfg.test_size,
        random_state=train_cfg.random_state,
    )
    x_test = x_test.cpu()
    train_loader, valid_loader = dataset.get_data_loaders(
        x_train,
        y_train,
        x_valid,
        y_valid,
        train_cfg,
    )    
    # dataloader = torch.utils.data.DataLoader(train_dataset, load_batch_size, shuffle=True, num_workers=4)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    device_type = torch.device(device).type

    model = MAE_ViT(
        image_size=256,
        mask_ratio=mask_ratio,
        patch_size=4,
    ).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=base_learning_rate * batch_size / 256, betas=(0.9, 0.95), weight_decay=weight_decay)
    lr_func = lambda epoch: min((epoch + 1) / (warmup_epoch + 1e-8), 0.5 * (math.cos(epoch / total_epoch * math.pi) + 1))
    lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda=lr_func)

    eval_model(model, valid_loader)
    step_count = 0
    for e in range(total_epoch):
        model.train()
        losses = []
        for img, label in tqdm(iter(train_loader)):
            optim.zero_grad()
            step_count += 1
            img = (img.to(device) - 0.5) / 0.5
            with torch.autocast(device_type, torch.bfloat16):
                predicted_img, mask = model(img)
                loss = torch.mean((predicted_img - img) ** 2 * mask) / mask_ratio
            loss.backward()
            if step_count % steps_per_update == 0:
                optim.step()
            losses.append(loss.item())
        lr_scheduler.step()
        avg_loss = sum(losses) / len(losses)
        print(f'In epoch {e}, average traning loss is {avg_loss}.')
        
        ''' save model '''
        torch.save(model, model_path)
        eval_model(model, valid_loader)

/root/repos/raidium_challenge/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
